In [1]:
import os
import json
import cv2
import torch
import networkx as nx
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm

import torchvision.transforms as transforms
import torchvision.models as models

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

from torch.nn import Linear

import torch.nn.functional as F

In [2]:
dataset_df = pd.read_csv(
    "../data/processed/full_composition_dataset.csv"
)

IMAGE_DIR = "../data/raw/CADB/images"

In [3]:
transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[0.485, 0.456, 0.406],

        std=[0.229, 0.224, 0.225]

    )
])

In [4]:
cnn_backbone = models.resnet18(
    weights="DEFAULT"
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\bhanu/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:14<00:00, 3.14MB/s]


In [5]:
cnn_backbone.fc = torch.nn.Identity()

In [38]:
from torch_geometric.data import Data

In [39]:
class HybridData(Data):

    def __cat_dim__(self, key, value, *args, **kwargs):

        if key == "image":

            return None

        return super().__cat_dim__(
            key,
            value,
            *args,
            **kwargs
        )

In [40]:
def create_hybrid_sample(

    image_path,
    graph_path,
    target

):

    # LOAD IMAGE

    img = cv2.imread(image_path)

    if img is None:
        return None

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    image_tensor = transform(img)

    # LOAD GRAPH

    with open(graph_path, "r") as f:

        graph_data = json.load(f)

    G = nx.node_link_graph(graph_data)

    # NODE FEATURES

    node_features = []

    for node, attrs in G.nodes(data=True):

        feature_vector = [

            attrs["center_x"],
            attrs["center_y"],
            attrs["area"],
            attrs["confidence"]

        ]

        node_features.append(feature_vector)

    # HANDLE EMPTY GRAPHS

    if len(node_features) == 0:

        return None

    # CREATE NODE FEATURE TENSOR

    x = torch.tensor(
        node_features,
        dtype=torch.float
    )

    # CREATE EDGE INDEX

    edge_index = []

    for u, v in G.edges():

        edge_index.append([u, v])
        edge_index.append([v, u])

    # HANDLE GRAPHS WITH NO EDGES
    # ADD SELF LOOPS

    if len(edge_index) == 0:

        num_nodes = len(node_features)

        for i in range(num_nodes):

            edge_index.append([i, i])

    edge_index = torch.tensor(
        edge_index,
        dtype=torch.long
    ).t().contiguous()

    # CREATE PYG DATA OBJECT

    data = HybridData(

        x=x,

        edge_index=edge_index,

        y=torch.tensor(
            [target],
            dtype=torch.float
        )
    )

    # ADD IMAGE

    data.image = image_tensor

    return data

In [41]:
hybrid_dataset = []

In [42]:
subset = dataset_df.head(200)

In [43]:
subset = dataset_df

In [44]:
for idx, row in tqdm(subset.iterrows()):

    image_name = row["image_name"]

    image_path = os.path.join(
        IMAGE_DIR,
        image_name
    )

    graph_path = (
        "../data/processed/graphs/"
        + image_name
        + ".json"
    )

    if not os.path.exists(graph_path):
        continue

    try:

        target = row["edge_density"]

        sample = create_hybrid_sample(

            image_path,
            graph_path,
            target

        )

        if sample is not None:

            hybrid_dataset.append(sample)

    except Exception as e:

        print(image_name, e)

9497it [03:07, 50.75it/s]


In [45]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(

    hybrid_dataset,

    test_size=0.2,

    random_state=42
)

In [46]:
train_loader = DataLoader(

    train_data,

    batch_size=8,

    shuffle=True
)

test_loader = DataLoader(

    test_data,

    batch_size=8
)

In [47]:
class HybridComposeNet(torch.nn.Module):

    def __init__(self):

        super().__init__()

        # CNN

        self.cnn = cnn_backbone

        # GNN

        self.conv1 = GCNConv(4, 32)

        self.conv2 = GCNConv(32, 64)

        # Fusion

        self.fc1 = Linear(512 + 64, 128)

        self.fc2 = Linear(128, 64)

        self.fc3 = Linear(64, 1)
    def forward(self, data):
    

        # IMAGE BRANCH

        images = data.image

        cnn_features = self.cnn(images)

        # GRAPH BRANCH

        x = data.x

        edge_index = data.edge_index

        batch = data.batch

        x = self.conv1(x, edge_index)

        x = F.relu(x)

        x = self.conv2(x, edge_index)

        x = F.relu(x)

        graph_features = global_mean_pool(
            x,
            batch
        )

        # FUSION

        combined = torch.cat(

            [cnn_features, graph_features],

            dim=1
        )

        x = self.fc1(combined)

        x = F.relu(x)

        x = self.fc2(x)

        x = F.relu(x)

        x = self.fc3(x)

        return x

In [48]:
device = torch.device(

    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = HybridComposeNet().to(device)

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.0005
)

criterion = torch.nn.MSELoss()

In [31]:
epochs = 10

In [49]:
for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        output = model(batch)

        loss = criterion(

            output.squeeze(),

            batch.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(

        f"Epoch {epoch+1} | Loss: {avg_loss:.4f}"

    )

Epoch 1 | Loss: 13516.3968
Epoch 2 | Loss: 1043.6441
Epoch 3 | Loss: 0.2202
Epoch 4 | Loss: 139.1452
Epoch 5 | Loss: 0.0492
Epoch 6 | Loss: 0.4186
Epoch 7 | Loss: 148.6252
Epoch 8 | Loss: 0.2388
Epoch 9 | Loss: 3.1427
Epoch 10 | Loss: 10.6856


In [50]:
model.eval()

predictions = []
targets = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        output = model(batch)

        predictions.extend(

            output.squeeze().cpu().numpy()

        )

        targets.extend(

            batch.y.cpu().numpy()
        )

In [51]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(
    targets,
    predictions
)

print("Hybrid Model MSE:", mse)

Hybrid Model MSE: 3.421985555971622


In [52]:
torch.save(

    model.state_dict(),

    "../checkpoints/hybrid_compose_net.pth"
)

print("Model saved successfully")

Model saved successfully
